# Análise por IA (Embeddings) — MRE 2024
**Opção 2 — Semântica contextual.** Cada parágrafo com `internet` é convertido em vetor e comparado aos centr óides `Soberana/Multilateral` vs `Mercado/Inovação`.
PDF: `relatorios-gestao-mre/Relatorio-Gestao-MRE-2024.pdf` | Modelo: `config.MODELO_EMBEDDING`


In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd()))
from config import (MODELO_EMBEDDING, TERMO_CENTRAL_REGEX, ABORDAGEM_A_TERMOS, ABORDAGEM_B_TERMOS,
                     ABORDAGEM_A_FRASES, ABORDAGEM_B_FRASES, ABORDAGEM_A_NOME, ABORDAGEM_B_NOME,
                     CHUNK_TIPO, CHUNK_TAMANHO, CHUNK_OVERLAP, FILTRAR_SOMENTE_COM_TERMO)
from utils import extract_text, chunk_por_paragrafo, chunk_janela_deslizante, filtrar_chunks_com_termo
from utils import carregar_modelo, gerar_centroides, embed_chunks, classificar_chunks

ANO=2024
PDF_FILE="Relatorio-Gestao-MRE-2024.pdf"
PDF_PATH=Path("/workspaces/governanca-digital_mre/relatorios-gestao-mre")/PDF_FILE
print(f"Ano {ANO} — {PDF_PATH} existe={PDF_PATH.exists()}")


In [ ]:
# 1. Extrair texto e chunking
texto = extract_text(PDF_PATH)
print(f"Caracteres: {len(texto):,}")
if CHUNK_TIPO == "paragrafo":
    chunks = chunk_por_paragrafo(texto)
else:
    chunks = chunk_janela_deslizante(texto, CHUNK_TAMANHO, CHUNK_OVERLAP)
print(f"Chunks totais: {len(chunks)}")
print(f"Exemplo chunk 0: {chunks[0][:300]}..." if chunks else "vazio")
filtrados = filtrar_chunks_com_termo(chunks, TERMO_CENTRAL_REGEX) if FILTRAR_SOMENTE_COM_TERMO else [{"idx_original":i, "texto":c} for i,c in enumerate(chunks)]
print(f"Chunks com 'internet': {len(filtrados)}")
for f in filtrados[:2]:
    print(f" - [{f['idx_original']}] {f['texto'][:250]}...")


In [ ]:
# 2. Carregar modelo e gerar centr óides A/B
# Requer: pip install sentence-transformers torch
try:
    model = carregar_modelo(MODELO_EMBEDDING)
    centroide_A, centroide_B = gerar_centroides(model, ABORDAGEM_A_TERMOS, ABORDAGEM_B_TERMOS, ABORDAGEM_A_FRASES, ABORDAGEM_B_FRASES)
    print(f"Centr óide A ({ABORDAGEM_A_NOME}): {centroide_A[:5]}... norm={sum(centroide_A**2)**0.5:.3f}")
    print(f"Centr óide B ({ABORDAGEM_B_NOME}): {centroide_B[:5]}... norm={sum(centroide_B**2)**0.5:.3f}")
    modelo_ok=True
except Exception as e:
    print(f"Modelo não carregado: {e}")
    print("Instale deps: pip install -r requirements.txt")
    modelo_ok=False


In [ ]:
# 3. Embeddings dos chunks filtrados + classificação
if not filtrados:
    print("Sem chunks com 'internet' — nada a classificar neste ano.")
elif not modelo_ok:
    print("Pule esta célula até instalar sentence-transformers.")
else:
    textos_filtrados = [f['texto'] for f in filtrados]
    embs = embed_chunks(model, textos_filtrados)
    resultados = classificar_chunks(embs, centroide_A, centroide_B)
    for f, r in zip(filtrados, resultados):
        f.update(r)
    from collections import Counter
    cnt = Counter(r['vencedor'] for r in resultados)
    print(f"Resumo {ANO}: {dict(cnt)}")
    try:
        import numpy as np
        print(f"Média sim_A={np.mean([r['sim_A'] for r in resultados]):.3f} sim_B={np.mean([r['sim_B'] for r in resultados]):.3f} diff={np.mean([r['diff'] for r in resultados]):.3f}")
    except ImportError:
        # fallback sem numpy
        mA=sum(r['sim_A'] for r in resultados)/len(resultados)
        mB=sum(r['sim_B'] for r in resultados)/len(resultados)
        md=sum(r['diff'] for r in resultados)/len(resultados)
        print(f"Média sim_A={mA:.3f} sim_B={mB:.3f} diff={md:.3f}")
    for f in sorted(filtrados, key=lambda x: x['diff'], reverse=True)[:2]:
        print(f"\n[TOP A] diff={f['diff']:.3f} simA={f['sim_A']:.3f} simB={f['sim_B']:.3f}")
        print(f['texto'][:400])
    for f in sorted(filtrados, key=lambda x: x['diff'])[:2]:
        print(f"\n[TOP B] diff={f['diff']:.3f} simA={f['sim_A']:.3f} simB={f['sim_B']:.3f}")
        print(f['texto'][:400])


In [ ]:
# 4. Visualização
if not filtrados or not modelo_ok:
    print("Sem dados para plotar.")
else:
    try:
        import matplotlib.pyplot as plt
        from collections import Counter
        cnt = Counter(r['vencedor'] for r in filtrados)
        labels = list(cnt.keys())
        vals = [cnt[k] for k in labels]
        plt.figure(figsize=(6,3))
        plt.bar(labels, vals)
        plt.title(f"MRE {ANO}: chunks com 'internet' por enquadramento (IA)")
        plt.ylabel("n chunks")
        plt.show()
        # Dispersão sim_A vs sim_B
        plt.figure(figsize=(5,5))
        for f in filtrados:
            c = 'purple' if f['vencedor']=='A' else 'orange' if f['vencedor']=='B' else 'gray'
            plt.scatter(f['sim_A'], f['sim_B'], c=c, alpha=0.7)
        plt.plot([0,1],[0,1],'--', color='gray', alpha=0.5)
        plt.xlabel(f"sim_{ABORDAGEM_A_NOME}")
        plt.ylabel(f"sim_{ABORDAGEM_B_NOME}")
        plt.title("Similaridade por chunk (diagonal = empate)")
        plt.show()
    except ImportError as e:
        print(f"sem matplotlib: {e}")


## Interpretação
- `diff > 0` => mais próximo de **A (Soberana)**; `diff < 0` => **B (Mercado)**; `|diff|<0.02` => neutro.
- Compare `TOP A/B` para ler trechos-âncora e validar qualitativamente.
- Para comparar anos, use `analise_comparativa.ipynb`.
- Ajuste léxicos/frases-âncora em `config.py` e re-execute.
